In [75]:
import os
import sys
import pandas as pd
from tsfresh.utilities.dataframe_functions import (
    roll_time_series,
)
from tsfresh import extract_features, select_features
from tsfresh.utilities.dataframe_functions import impute
from tsfresh.feature_extraction import ComprehensiveFCParameters
from sklearn.linear_model import LinearRegression
import matplotlib.pylab as plt

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from logger.logger import Logger
from utils.constants import FEATURE_SELECTION_LOG_FILE_BASE
from tabular_database_driver.postgre_sql_driver import PostgreSQLDriver
from dtos.tabular_database_driver_dtos.postgre_sql_connection_dto import (
    PostgreSQLConnectionDto,
)

In [44]:
my_logger = Logger(file_name=f"{FEATURE_SELECTION_LOG_FILE_BASE}/vn_index/test")

In [45]:
my_connection_model = PostgreSQLConnectionDto(
    logger=my_logger,
    host=os.getenv("POSTGRES_HOST"),
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
    port=os.getenv("POSTGRES_PORT"),
    database=os.getenv("GOLD_POSTGRES_DATABASE"),
)

In [46]:
my_postgresql_driver = PostgreSQLDriver(logger=my_logger)
my_postgresql_driver.connect(my_connection_model)

<DatabaseExecutionStatus.SUCCESS: 'success'>

In [47]:
vn_index_df = my_postgresql_driver.select(
    schema_name="stock_market", table_name="vn_index"
)

In [48]:
dtype_map = {
    "date": str,
    "open": float,
    "high": float,
    "low": float,
    "close": float,
    "volume": float,
}

vn_index_df = (
    vn_index_df.astype(dtype_map)
    .dropna(subset=["close"])
    .sort_values(by=["date"])
    .reset_index(drop=True)
)

In [49]:
vn_index_df

,date,open,high,low,close,volume
0,2000-07-28,100.000000,100.000000,100.000000,100.000000,4200.0
1,2000-07-29,100.516667,100.516667,100.516667,100.516667,6233.0
2,2000-07-30,101.033333,101.033333,101.033333,101.033333,8267.0
3,2000-07-31,101.550000,101.550000,101.550000,101.550000,10300.0
4,2000-08-01,102.465000,102.465000,102.465000,102.465000,5300.0
...,...,...,...,...,...,...
9099,2025-06-26,1368.730000,1370.610000,1360.780000,1365.670000,598388700.0
9100,2025-06-27,1369.130000,1373.280000,1362.090000,1371.440000,677408300.0
9101,2025-06-28,1371.340000,1374.620000,1365.403333,1372.983333,663601667.0
9102,2025-06-29,1373.550000,1375.960000,1368.716667,1374.526667,649795033.0


In [50]:
vn_index_df_close = vn_index_df[["date", "close"]]
vn_index_df_close

,date,close
0,2000-07-28,100.000000
1,2000-07-29,100.516667
2,2000-07-30,101.033333
3,2000-07-31,101.550000
4,2000-08-01,102.465000
...,...,...
9099,2025-06-26,1365.670000
9100,2025-06-27,1371.440000
9101,2025-06-28,1372.983333
9102,2025-06-29,1374.526667


In [51]:
vn_index_df_close.loc[:, "stock"] = "vn_index"
vn_index_df_close

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_22988\2662759176.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  vn_index_df_close.loc[:, "stock"] = "vn_index"


,date,close,stock
0,2000-07-28,100.000000,vn_index
1,2000-07-29,100.516667,vn_index
2,2000-07-30,101.033333,vn_index
3,2000-07-31,101.550000,vn_index
4,2000-08-01,102.465000,vn_index
...,...,...,...
9099,2025-06-26,1365.670000,vn_index
9100,2025-06-27,1371.440000,vn_index
9101,2025-06-28,1372.983333,vn_index
9102,2025-06-29,1374.526667,vn_index


In [52]:
vn_index_df_close_rolled = roll_time_series(
    vn_index_df_close,
    column_id="stock",
    column_sort="date",
    max_timeshift=20,
    min_timeshift=5,
)
vn_index_df_close_rolled

Rolling: 100%|██████████| 50/50 [00:06<00:00,  8.09it/s]


,date,close,stock,id
0,2000-07-28,100.000000,vn_index,"(vn_index, 2000-08-02)"
1,2000-07-29,100.516667,vn_index,"(vn_index, 2000-08-02)"
2,2000-07-30,101.033333,vn_index,"(vn_index, 2000-08-02)"
3,2000-07-31,101.550000,vn_index,"(vn_index, 2000-08-02)"
4,2000-08-01,102.465000,vn_index,"(vn_index, 2000-08-02)"
...,...,...,...,...
183268,2025-06-26,1365.670000,vn_index,"(vn_index, 2025-06-30)"
183269,2025-06-27,1371.440000,vn_index,"(vn_index, 2025-06-30)"
183270,2025-06-28,1372.983333,vn_index,"(vn_index, 2025-06-30)"
183271,2025-06-29,1374.526667,vn_index,"(vn_index, 2025-06-30)"


In [53]:
X = extract_features(
    vn_index_df_close_rolled.drop("stock", axis=1),
    column_id="id",
    column_sort="date",
    column_value="close",
    impute_function=impute,
    default_fc_parameters=ComprehensiveFCParameters(),
)

Feature Extraction: 100%|██████████| 50/50 [01:14<00:00,  1.50s/it]


In [54]:
X

close__variance_larger_than_standard_deviation  \
vn_index 2000-08-02                                             1.0   
         2000-08-03                                             1.0   
         2000-08-04                                             1.0   
         2000-08-05                                             1.0   
         2000-08-06                                             1.0   
...                                                             ...   
         2025-06-26                                             1.0   
         2025-06-27                                             1.0   
         2025-06-28                                             1.0   
         2025-06-29                                             1.0   
         2025-06-30                                             1.0   

                     close__has_duplicate_max  close__has_duplicate_min  \
vn_index 2000-08-02                       0.0                       0.0   
         2000-08-03                       0.0                       0.0   
         2000-08-04                       0.0                       0.0   
         2000-08-05                       0.0                       0.0   
         2000-08-06                       0.0                       0.0   
...                                       ...                       ...   
         2025-06-26                       0.0                       0.0   
         2025-06-27                       0.0                       0.0   
         2025-06-28                       0.0                       0.0   
         2025-06-29                       0.0                       0.0   
         2025-06-30                       0.0                       0.0   

                     close__has_duplicate  close__sum_values  \
vn_index 2000-08-02                   0.0         608.945000   
         2000-08-03                   0.0         713.235000   
         2000-08-04                   0.0         818.435000   
         2000-08-05                   0.0         924.208333   
         2000-08-06                   0.0        1030.555000   
...                                   ...                ...   
         2025-06-26                   0.0       28103.350000   
         2025-06-27                   0.0       28144.900000   
         2025-06-28                   0.0       28194.433333   
         2025-06-29                   0.0       28251.950000   
         2025-06-30                   0.0       28317.450000   

                     close__abs_energy  close__mean_abs_change  \
vn_index 2000-08-02       6.181024e+04                0.676000   
         2000-08-03       7.268664e+04                0.715000   
         2000-08-04       8.375368e+04                0.742857   
         2000-08-05       9.494168e+04                0.721667   
         2000-08-06       1.062513e+05                0.705185   
...                                ...                     ...   
         2025-06-26       3.761674e+07                5.039000   
         2025-06-27       3.772898e+07                5.005500   
         2025-06-28       3.786255e+07                4.760667   
         2025-06-29       3.801735e+07                4.515833   
         2025-06-30       3.819333e+07                4.310000   

                     close__mean_change  \
vn_index 2000-08-02            0.676000   
         2000-08-03            0.715000   
         2000-08-04            0.742857   
         2000-08-05            0.721667   
         2000-08-06            0.705185   
...                                 ...   
         2025-06-26            1.789000   
         2025-06-27            2.399500   
         2025-06-28            2.798667   
         2025-06-29            3.197833   
         2025-06-30            2.992000   

                     close__mean_second_derivative_central  close__median  \
vn_index 2000-08-02                               0.049792     101.291667   
         2000-08-03               

In [55]:
X = X.set_index(X.index.map(lambda x: x[1]), drop=True)
X.index.name = "last_date"
X

,close__variance_larger_than_standard_deviation,close__has_duplicate_max,close__has_duplicate_min,close__has_duplicate,close__sum_values,close__abs_energy,close__mean_abs_change,close__mean_change,close__mean_second_derivative_central,close__median,...,close__fourier_entropy__bins_5,close__fourier_entropy__bins_10,close__fourier_entropy__bins_100,close__permutation_entropy__dimension_3__tau_1,close__permutation_entropy__dimension_4__tau_1,close__permutation_entropy__dimension_5__tau_1,close__permutation_entropy__dimension_6__tau_1,close__permutation_entropy__dimension_7__tau_1,close__query_similarity_count__query_None__threshold_0.0,close__mean_n_absolute_max__number_of_maxima_7
last_date,,,,,,,,,,,,,,,,,,,,,
2000-08-02,1.0,0.0,0.0,0.0,608.945000,6.181024e+04,0.676000,0.676000,0.049792,101.291667,...,0.562335,0.562335,1.386294,-0.000000,-0.000000,-0.000000,-0.000000,2.615631,0.0,572.868571
2000-08-03,1.0,0.0,0.0,0.0,713.235000,7.268664e+04,0.715000,0.715000,0.039333,101.550000,...,0.562335,0.562335,1.386294,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.0,572.868571
2000-08-04,1.0,0.0,0.0,0.0,818.435000,8.375368e+04,0.742857,0.742857,0.032778,102.007500,...,0.500402,0.500402,1.332179,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.0,102.633571
2000-08-05,1.0,0.0,0.0,0.0,924.208333,9.494168e+04,0.721667,0.721667,0.004048,102.465000,...,0.500402,0.500402,1.332179,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.0,103.384524
2000-08-06,1.0,0.0,0.0,0.0,1030.555000,1.062513e+05,0.705185,0.705185,0.003542,102.922500,...,0.450561,0.450561,1.242453,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.0,104.143571
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-06-26,1.0,0.0,0.0,0.0,28103.350000,3.761674e+07,5.039000,1.789000,0.141053,1338.110000,...,0.304636,0.600166,1.294545,1.489767,2.120208,2.507026,2.512659,2.523211,0.0,1359.562857
2025-06-27,1.0,0.0,0.0,0.0,28144.900000,3.772898e+07,5.005500,2.399500,0.321316,1346.830000,...,0.304636,0.600166,1.159589,1.458585,2.120208,2.507026,2.512659,2.523211,0.0,1362.334286
2025-06-28,1.0,0.0,0.0,0.0,28194.433333,3.786255e+07,4.760667,2.798667,0.210088,1347.690000,...,0.304636,0.304636,1.294545,1.380452,2.014123,2.507026,2.512659,2.523211,0.0,1365.290000


In [60]:
y = vn_index_df_close.set_index("date").sort_index()["close"].shift(-1)
y

date
2000-07-28     100.516667
2000-07-29     101.033333
2000-07-30     101.550000
2000-07-31     102.465000
2000-08-01     103.380000
                 ...     
2025-06-26    1371.440000
2025-06-27    1372.983333
2025-06-28    1374.526667
2025-06-29    1376.070000
2025-06-30            NaN
Name: close, Length: 9104, dtype: float64

In [61]:
y = y[y.index.isin(X.index)]
X = X[X.index.isin(y.index)]

In [62]:
X

,close__variance_larger_than_standard_deviation,close__has_duplicate_max,close__has_duplicate_min,close__has_duplicate,close__sum_values,close__abs_energy,close__mean_abs_change,close__mean_change,close__mean_second_derivative_central,close__median,...,close__fourier_entropy__bins_5,close__fourier_entropy__bins_10,close__fourier_entropy__bins_100,close__permutation_entropy__dimension_3__tau_1,close__permutation_entropy__dimension_4__tau_1,close__permutation_entropy__dimension_5__tau_1,close__permutation_entropy__dimension_6__tau_1,close__permutation_entropy__dimension_7__tau_1,close__query_similarity_count__query_None__threshold_0.0,close__mean_n_absolute_max__number_of_maxima_7
last_date,,,,,,,,,,,,,,,,,,,,,
2000-08-02,1.0,0.0,0.0,0.0,608.945000,6.181024e+04,0.676000,0.676000,0.049792,101.291667,...,0.562335,0.562335,1.386294,-0.000000,-0.000000,-0.000000,-0.000000,2.615631,0.0,572.868571
2000-08-03,1.0,0.0,0.0,0.0,713.235000,7.268664e+04,0.715000,0.715000,0.039333,101.550000,...,0.562335,0.562335,1.386294,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.0,572.868571
2000-08-04,1.0,0.0,0.0,0.0,818.435000,8.375368e+04,0.742857,0.742857,0.032778,102.007500,...,0.500402,0.500402,1.332179,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.0,102.633571
2000-08-05,1.0,0.0,0.0,0.0,924.208333,9.494168e+04,0.721667,0.721667,0.004048,102.465000,...,0.500402,0.500402,1.332179,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.0,103.384524
2000-08-06,1.0,0.0,0.0,0.0,1030.555000,1.062513e+05,0.705185,0.705185,0.003542,102.922500,...,0.450561,0.450561,1.242453,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.0,104.143571
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-06-26,1.0,0.0,0.0,0.0,28103.350000,3.761674e+07,5.039000,1.789000,0.141053,1338.110000,...,0.304636,0.600166,1.294545,1.489767,2.120208,2.507026,2.512659,2.523211,0.0,1359.562857
2025-06-27,1.0,0.0,0.0,0.0,28144.900000,3.772898e+07,5.005500,2.399500,0.321316,1346.830000,...,0.304636,0.600166,1.159589,1.458585,2.120208,2.507026,2.512659,2.523211,0.0,1362.334286
2025-06-28,1.0,0.0,0.0,0.0,28194.433333,3.786255e+07,4.760667,2.798667,0.210088,1347.690000,...,0.304636,0.304636,1.294545,1.380452,2.014123,2.507026,2.512659,2.523211,0.0,1365.290000


In [63]:
y

date
2000-08-02     104.290000
2000-08-03     105.200000
2000-08-04     105.773333
2000-08-05     106.346667
2000-08-06     106.920000
                 ...     
2025-06-26    1371.440000
2025-06-27    1372.983333
2025-06-28    1374.526667
2025-06-29    1376.070000
2025-06-30            NaN
Name: close, Length: 9099, dtype: float64

In [64]:
X_train = X[:"2025"]
X_train

,close__variance_larger_than_standard_deviation,close__has_duplicate_max,close__has_duplicate_min,close__has_duplicate,close__sum_values,close__abs_energy,close__mean_abs_change,close__mean_change,close__mean_second_derivative_central,close__median,...,close__fourier_entropy__bins_5,close__fourier_entropy__bins_10,close__fourier_entropy__bins_100,close__permutation_entropy__dimension_3__tau_1,close__permutation_entropy__dimension_4__tau_1,close__permutation_entropy__dimension_5__tau_1,close__permutation_entropy__dimension_6__tau_1,close__permutation_entropy__dimension_7__tau_1,close__query_similarity_count__query_None__threshold_0.0,close__mean_n_absolute_max__number_of_maxima_7
last_date,,,,,,,,,,,,,,,,,,,,,
2000-08-02,1.0,0.0,0.0,0.0,608.945000,6.181024e+04,0.676000,0.676000,0.049792,101.291667,...,0.562335,0.562335,1.386294,-0.000000,-0.000000,-0.000000,-0.000000,2.615631,0.0,572.868571
2000-08-03,1.0,0.0,0.0,0.0,713.235000,7.268664e+04,0.715000,0.715000,0.039333,101.550000,...,0.562335,0.562335,1.386294,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.0,572.868571
2000-08-04,1.0,0.0,0.0,0.0,818.435000,8.375368e+04,0.742857,0.742857,0.032778,102.007500,...,0.500402,0.500402,1.332179,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.0,102.633571
2000-08-05,1.0,0.0,0.0,0.0,924.208333,9.494168e+04,0.721667,0.721667,0.004048,102.465000,...,0.500402,0.500402,1.332179,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.0,103.384524
2000-08-06,1.0,0.0,0.0,0.0,1030.555000,1.062513e+05,0.705185,0.705185,0.003542,102.922500,...,0.450561,0.450561,1.242453,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.0,104.143571
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-27,1.0,0.0,0.0,0.0,26584.140000,3.365391e+07,3.012333,0.188333,0.027281,1263.790000,...,0.600166,0.885574,2.019815,1.666876,2.399204,2.751667,2.772589,2.708050,0.0,1273.134286
2024-12-28,1.0,0.0,0.0,0.0,26586.866667,3.366085e+07,3.002667,0.074667,-0.059825,1263.790000,...,0.885574,0.885574,2.145842,1.736195,2.399204,2.751667,2.772589,2.708050,0.0,1273.523810
2024-12-29,1.0,0.0,0.0,0.0,26587.320000,3.366200e+07,2.993000,-0.039000,0.019211,1263.790000,...,0.885574,1.159589,2.145842,1.736195,2.399204,2.751667,2.772589,2.708050,0.0,1273.588571


In [65]:
X_test = X["2025":]
X_test

,close__variance_larger_than_standard_deviation,close__has_duplicate_max,close__has_duplicate_min,close__has_duplicate,close__sum_values,close__abs_energy,close__mean_abs_change,close__mean_change,close__mean_second_derivative_central,close__median,...,close__fourier_entropy__bins_5,close__fourier_entropy__bins_10,close__fourier_entropy__bins_100,close__permutation_entropy__dimension_3__tau_1,close__permutation_entropy__dimension_4__tau_1,close__permutation_entropy__dimension_5__tau_1,close__permutation_entropy__dimension_6__tau_1,close__permutation_entropy__dimension_7__tau_1,close__query_similarity_count__query_None__threshold_0.0,close__mean_n_absolute_max__number_of_maxima_7
last_date,,,,,,,,,,,,,,,,,,,,,
2025-01-01,1.0,0.0,0.0,0.0,26579.595000,3.364238e+07,3.055750,0.044750,0.164342,1263.79,...,0.600166,0.885574,1.972247,1.754079,2.399204,2.751667,2.772589,2.708050,0.0,1272.782143
2025-01-02,1.0,0.0,0.0,0.0,26581.955000,3.364836e+07,2.890000,0.357000,0.027851,1263.79,...,0.600166,0.885574,2.019815,1.712299,2.399204,2.833213,2.772589,2.708050,0.0,1272.991429
2025-01-03,1.0,0.0,0.0,0.0,26573.975000,3.362828e+07,3.625667,-0.419333,-0.408596,1263.79,...,0.600166,0.600166,1.972247,1.736195,2.428274,2.833213,2.772589,2.708050,0.0,1272.991429
2025-01-04,1.0,0.0,0.0,0.0,26562.841667,3.360028e+07,3.742667,-0.577000,-0.082982,1263.79,...,0.600166,0.885574,2.145842,1.749494,2.505290,2.833213,2.772589,2.708050,0.0,1272.991429
2025-01-05,1.0,0.0,0.0,0.0,26548.555000,3.356438e+07,3.859667,-0.734667,-0.017807,1263.79,...,0.885574,0.885574,1.767761,1.736195,2.476221,2.833213,2.772589,2.708050,0.0,1272.991429
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-06-26,1.0,0.0,0.0,0.0,28103.350000,3.761674e+07,5.039000,1.789000,0.141053,1338.11,...,0.304636,0.600166,1.294545,1.489767,2.120208,2.507026,2.512659,2.523211,0.0,1359.562857
2025-06-27,1.0,0.0,0.0,0.0,28144.900000,3.772898e+07,5.005500,2.399500,0.321316,1346.83,...,0.304636,0.600166,1.159589,1.458585,2.120208,2.507026,2.512659,2.523211,0.0,1362.334286
2025-06-28,1.0,0.0,0.0,0.0,28194.433333,3.786255e+07,4.760667,2.798667,0.210088,1347.69,...,0.304636,0.304636,1.294545,1.380452,2.014123,2.507026,2.512659,2.523211,0.0,1365.290000


In [66]:
y_train = y[:"2025"]
y_train

date
2000-08-02     104.290000
2000-08-03     105.200000
2000-08-04     105.773333
2000-08-05     106.346667
2000-08-06     106.920000
                 ...     
2024-12-27    1274.100000
2024-12-28    1273.060000
2024-12-29    1272.020000
2024-12-30    1266.780000
2024-12-31    1268.245000
Name: close, Length: 8918, dtype: float64

In [67]:
y_test = y["2025":]
y_test

date
2025-01-01    1269.710000
2025-01-02    1254.590000
2025-01-03    1251.843333
2025-01-04    1249.096667
2025-01-05    1246.350000
                 ...     
2025-06-26    1371.440000
2025-06-27    1372.983333
2025-06-28    1374.526667
2025-06-29    1376.070000
2025-06-30            NaN
Name: close, Length: 181, dtype: float64

In [69]:
X_train_selected = select_features(X_train, y_train)
X_train_selected

,close__sum_values,close__absolute_maximum,close__maximum,close__minimum,close__mean_n_absolute_max__number_of_maxima_7,close__quantile__q_0.8,close__quantile__q_0.7,close__quantile__q_0.6,close__quantile__q_0.1,close__quantile__q_0.2,...,"close__cwt_coefficients__coeff_10__w_2__widths_(2, 5, 10, 20)",close__energy_ratio_by_chunks__num_segments_10__segment_focus_5,"close__fft_coefficient__attr_""angle""__coeff_9",close__ratio_beyond_r_sigma__r_2.5,"close__fft_coefficient__attr_""imag""__coeff_10","close__augmented_dickey_fuller__attr_""usedlag""__autolag_""AIC""",close__partial_autocorrelation__lag_4,close__energy_ratio_by_chunks__num_segments_10__segment_focus_2,close__large_standard_deviation__r_0.2,close__large_standard_deviation__r_0.30000000000000004
last_date,,,,,,,,,,,,,,,,,,,,,
2000-08-02,608.945000,103.380000,103.380000,100.00,572.868571,102.465000,102.007500,101.550,100.258333,100.516667,...,0.190537,0.172907,10.292836,0.0,0.172690,0.0,-0.143933,0.165146,1.0,1.0
2000-08-03,713.235000,104.290000,104.290000,100.00,572.868571,103.197000,102.648000,102.099,100.310000,100.620000,...,0.190537,0.147034,10.292836,0.0,0.172690,0.0,-0.143933,0.140435,1.0,1.0
2000-08-04,818.435000,105.200000,105.200000,100.00,102.633571,103.926000,103.288500,102.648,100.361667,100.723333,...,0.190537,0.127605,10.292836,0.0,0.172690,0.0,-0.143933,0.121878,1.0,1.0
2000-08-05,924.208333,105.773333,105.773333,100.00,103.384524,104.654000,103.926000,103.197,100.413333,100.826667,...,0.190537,0.112568,10.292836,0.0,0.172690,0.0,-0.143933,0.107516,1.0,1.0
2000-08-06,1030.555000,106.346667,106.346667,100.00,104.143571,105.314667,104.563000,103.744,100.465000,100.930000,...,0.190537,0.100586,10.292836,0.0,0.172690,2.0,-0.525011,0.096072,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-27,26584.140000,1275.140000,1275.140000,1254.67,1273.134286,1272.606667,1271.373333,1267.350,1259.253333,1261.006667,...,2.395897,0.094401,99.501239,0.0,-6.500507,0.0,0.010131,0.095093,1.0,0.0
2024-12-28,26586.866667,1275.140000,1275.140000,1254.67,1273.523810,1272.870000,1272.070000,1267.350,1259.253333,1261.006667,...,0.987972,0.093744,-114.850325,0.0,5.120264,0.0,-0.279790,0.094745,1.0,0.0
2024-12-29,26587.320000,1275.140000,1275.140000,1254.67,1273.588571,1273.060000,1272.070000,1267.350,1259.253333,1261.006667,...,-3.666938,0.094083,40.765492,0.0,-3.558076,0.0,-0.239258,0.094803,1.0,0.0
